In [1]:
import os
import sys

sys.path.append(os.path.dirname(os.getcwd())) 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from datetime import datetime, timedelta

from src.constants import hustle_stats, four_factors_stats, home_away_id_cols
from src.steps.model.regression_diagnostics import RegressionDiagnostics
from src.steps.evaluation.utils_evaluation import rmse

import logging
from typing import Optional, List
from statsmodels.regression.linear_model import RegressionResultsWrapper
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

pd.set_option('display.max_columns', None)

In [2]:
class TransformStage2:
    def __init__(self, df_transformed: pd.DataFrame, df_stage1_results: pd.DataFrame, current_date: str, hist_avg_window: int):
        self.df_transformed = df_transformed
        self.df_stage1_results = df_stage1_results
        self.current_date = current_date
        self.hist_avg_window = hist_avg_window
        self.df_team_sched = pd.DataFrame()

        self.current_date = pd.to_datetime(self.current_date)

    @staticmethod
    def team_schedule(df: pd.DataFrame) -> pd.DataFrame:
        cols = [
            'GAME_ID','HOME_TEAM','AWAY_TEAM','GAME_DATE',
            'HOME_EFG_PCT','AWAY_EFG_PCT', 
            'HOME_TM_TOV_PCT','AWAY_TM_TOV_PCT',
            'HOME_FTA_RATE','AWAY_FTA_RATE',
            'HOME_OREB_PCT','AWAY_OREB_PCT',
            'HOME_NET_COMPOSITE_EFFORT','AWAY_NET_COMPOSITE_EFFORT'
        ]
        df_home = (
            df[cols]
            .rename(columns={
                'HOME_TEAM':'TEAM',
                'AWAY_TEAM':'OPP_TEAM',
                'HOME_EFG_PCT':'TEAM_EFG_PCT',
                'AWAY_EFG_PCT':'OPP_EFG_PCT',
                'HOME_TM_TOV_PCT': 'TEAM_TM_TOV_PCT',
                'AWAY_TM_TOV_PCT':'OPP_TOV_PCT',
                'HOME_FTA_RATE':'TEAM_FTA_RATE',
                'AWAY_FTA_RATE':'OPP_FTA_RATE',
                'HOME_OREB_PCT':'TEAM_OREB_PCT',
                'AWAY_OREB_PCT':'OPP_OREB_PCT',
                'HOME_NET_COMPOSITE_EFFORT':'TEAM_NET_COMPOSITE_EFFORT',
                'AWAY_NET_COMPOSITE_EFFORT':'OPP_NET_COMPOSITE_EFFORT'
            })
            .assign(HOME_IND=1)
        )

        df_away = (
            df[cols]
            .rename(columns={
                'AWAY_TEAM':'TEAM',
                'HOME_TEAM':'OPP_TEAM',
                'HOME_EFG_PCT':'OPP_EFG_PCT',
                'AWAY_EFG_PCT':'TEAM_EFG_PCT',
                'HOME_TM_TOV_PCT': 'OPP_TOV_PCT',
                'AWAY_TM_TOV_PCT':'TEAM_TM_TOV_PCT',
                'HOME_FTA_RATE':'OPP_FTA_RATE',
                'AWAY_FTA_RATE':'TEAM_FTA_RATE',
                'HOME_OREB_PCT':'OPP_OREB_PCT',
                'AWAY_OREB_PCT':'TEAM_OREB_PCT',
                'HOME_NET_COMPOSITE_EFFORT':'OPP_NET_COMPOSITE_EFFORT',
                'AWAY_NET_COMPOSITE_EFFORT':'TEAM_NET_COMPOSITE_EFFORT'
            })
            .assign(HOME_IND=0)
        )
        team_schedule = (
            pd.concat([df_home, df_away], ignore_index=True)
            .sort_values(['TEAM','GAME_DATE'])
            .reset_index(drop=True)
        )

        # Create season for grouping
        team_schedule['GAME_ID_str'] = team_schedule['GAME_ID'].astype(str)
        team_schedule['SEASON_SUFFIX'] = team_schedule['GAME_ID_str'].str[1:3]
        team_schedule['SEASON'] = '20' + team_schedule['SEASON_SUFFIX']
        team_schedule['SEASON'] = team_schedule['SEASON'].astype(int)
        team_schedule.drop(['GAME_ID_str','SEASON_SUFFIX'], axis=1, inplace=True)
        team_schedule.drop(['OPP_EFG_PCT','OPP_TOV_PCT','OPP_FTA_RATE','OPP_OREB_PCT','OPP_NET_COMPOSITE_EFFORT'], axis=1, inplace=True)

        return team_schedule.sort_values(['TEAM','GAME_DATE'])
    
    def merge_stage1_results(self):
        
        #df_transformed = self.df_transformed.copy()
        #df_stage1_results = self.df_stage1_results.copy()

        # In two stages: 1. Merge home results, 2. Merge away results
        cols_to_drop = ['SEASON_ID','GAME_DATE','TEAM_ABBREVIATION']
        self.df_transformed = self.df_transformed.merge(
            self.df_stage1_results.drop(cols_to_drop,axis=1),
            how='left',
            left_on=['GAME_ID','HOME_TEAM_ID'],
            right_on=['GAME_ID','TEAM_ID']
        )
        self. df_transformed = self.df_transformed.merge(
            self.df_stage1_results.drop(cols_to_drop,axis=1),
            how='left',
            left_on=['GAME_ID','AWAY_TEAM_ID'],
            right_on=['GAME_ID','TEAM_ID'],
            suffixes=('_HOME','_AWAY')
        )

        # Drop TEAM_ID_HOME and TEAM_ID_AWAY to avoid redundancy
        self.df_transformed.drop(['TEAM_ID_HOME','TEAM_ID_AWAY'], axis=1, inplace=True)
    
    def clean_data(self):
        """Clean Columns and Split Data"""
        end_home_cols = self.df_transformed.columns[self.df_transformed.columns.str.endswith('_HOME')]
        end_away_cols = self.df_transformed.columns[self.df_transformed.columns.str.endswith('_AWAY')]
        begin_home_cols = [f"HOME_{col.replace('_HOME','')}" for col in end_home_cols]
        begin_away_cols = [f"AWAY_{col.replace('_AWAY','')}" for col in end_away_cols]
        home_cols = dict(zip(end_home_cols, begin_home_cols))
        away_cols = dict(zip(end_away_cols, begin_away_cols))
        new_cols = {**home_cols, **away_cols}
        self.df_transformed.rename(columns=new_cols, inplace=True)

        self.df_transformed.drop(['HOME_TEAM_NAME','AWAY_TEAM_NAME'], axis=1, inplace=True)
        self.df_transformed.rename(
            columns={
                'HOME_TEAM_ABBREVIATION':'HOME_TEAM',
                'AWAY_TEAM_ABBREVIATION':'AWAY_TEAM'
            }, 
            inplace=True
        )

        self.df_transformed['GAME_DATE'] = pd.to_datetime(self.df_transformed['GAME_DATE'])
        
        self.df_transformed = self.df_transformed[self.df_transformed['GAME_DATE'] <= self.current_date]
        # self.df_train = self.df_transformed[self.df_transformed['GAME_DATE'] < self.current_date].copy()
        # self.df_test = self.df_transformed[self.df_transformed['GAME_DATE'] == self.current_date].copy()

    
    def create_historical_features(self):
        """Four Factors and Effort Average"""
        self.df_team_sched = self.team_schedule(df=self.df_transformed)
        self.df_team_sched[f'TEAM_AVG{self.hist_avg_window}_NET_COMPOSITE_EFFORT'] = (
            self.df_team_sched
            .groupby('TEAM')['TEAM_NET_COMPOSITE_EFFORT']
            .rolling(window=self.hist_avg_window, min_periods=1, closed='left')
            .mean()
            .reset_index(drop=True)
        )
        for ff in four_factors_stats:
            self.df_team_sched[f"TEAM_AVG{self.hist_avg_window}_{ff}"] = (
                self.df_team_sched
                .groupby('TEAM')[f'TEAM_{ff}']
                .rolling(window=self.hist_avg_window, min_periods=1, closed='left')
                .mean()
                .reset_index(drop=True)
            )

    
    def join_historical_features(self):
        # Join back to transformed data: 1. join by home, 2. join by away 
        cols_to_keep = [
            'GAME_ID',
            'TEAM',
            'OPP_TEAM',
            f'TEAM_AVG{self.hist_avg_window}_NET_COMPOSITE_EFFORT',
            f'TEAM_AVG{self.hist_avg_window}_EFG_PCT',
            f'TEAM_AVG{self.hist_avg_window}_FTA_RATE',
            f'TEAM_AVG{self.hist_avg_window}_TM_TOV_PCT',
            f'TEAM_AVG{self.hist_avg_window}_OREB_PCT'
        ]
        df_team_sched = self.df_team_sched[cols_to_keep]
        self.df_transformed = (
            self.df_transformed
            .merge(
            df_team_sched, 
            how='left', 
            left_on=['GAME_ID','HOME_TEAM'],
            right_on=['GAME_ID','TEAM'])
            .rename(columns={
                f'TEAM_AVG{self.hist_avg_window}_NET_COMPOSITE_EFFORT':f'HOME_AVG{self.hist_avg_window}_NET_COMPOSITE_EFFORT',
                f'TEAM_AVG{self.hist_avg_window}_EFG_PCT':f'HOME_AVG{self.hist_avg_window}_EFG_PCT',
                f'TEAM_AVG{self.hist_avg_window}_FTA_RATE':f'HOME_AVG{self.hist_avg_window}_FTA_RATE',
                f'TEAM_AVG{self.hist_avg_window}_TM_TOV_PCT':f'HOME_AVG{self.hist_avg_window}_TM_TOV_PCT',
                f'TEAM_AVG{self.hist_avg_window}_OREB_PCT':f'HOME_AVG{self.hist_avg_window}_OREB_PCT'
                }
            )
            .drop(['TEAM','OPP_TEAM'],axis=1)
        )
        self.df_transformed = (
            self.df_transformed
            .merge(
                df_team_sched,
                how='left',
                left_on=['GAME_ID','AWAY_TEAM'],
                right_on=['GAME_ID','TEAM']
            )
            .rename(columns={
                f'TEAM_AVG{self.hist_avg_window}_NET_COMPOSITE_EFFORT':f'AWAY_AVG{self.hist_avg_window}_NET_COMPOSITE_EFFORT',
                f'TEAM_AVG{self.hist_avg_window}_EFG_PCT':f'AWAY_AVG{self.hist_avg_window}_EFG_PCT',
                f'TEAM_AVG{self.hist_avg_window}_FTA_RATE':f'AWAY_AVG{self.hist_avg_window}_FTA_RATE',
                f'TEAM_AVG{self.hist_avg_window}_TM_TOV_PCT':f'AWAY_AVG{self.hist_avg_window}_TM_TOV_PCT',
                f'TEAM_AVG{self.hist_avg_window}_OREB_PCT':f'AWAY_AVG{self.hist_avg_window}_OREB_PCT'
                }
            )
            .drop(['TEAM','OPP_TEAM'],axis=1)
            
        )

        # Create differenced features
        self.df_transformed[f'AVG{self.hist_avg_window}_NET_COMPOSITE_EFFORT_DIFF'] = self.df_transformed[f'HOME_AVG{self.hist_avg_window}_NET_COMPOSITE_EFFORT']-self.df_transformed[f'AWAY_AVG{self.hist_avg_window}_NET_COMPOSITE_EFFORT']
        self.df_transformed[f'AVG{self.hist_avg_window}_EFG_PCT_DIFF'] = self.df_transformed[f'HOME_AVG{self.hist_avg_window}_EFG_PCT']-self.df_transformed[f'AWAY_AVG{self.hist_avg_window}_EFG_PCT']
        self.df_transformed[f'AVG{self.hist_avg_window}_FTA_RATE_DIFF'] = self.df_transformed[f'HOME_AVG{self.hist_avg_window}_FTA_RATE']-self.df_transformed[f'AWAY_AVG{self.hist_avg_window}_FTA_RATE']
        self.df_transformed[f'AVG{self.hist_avg_window}_TM_TOV_PCT_DIFF'] = self.df_transformed[f'HOME_AVG{self.hist_avg_window}_TM_TOV_PCT']-self.df_transformed[f'AWAY_AVG{self.hist_avg_window}_TM_TOV_PCT']
        self.df_transformed[f'AVG{self.hist_avg_window}_OREB_PCT_DIFF'] = self.df_transformed[f'HOME_AVG{self.hist_avg_window}_OREB_PCT']-self.df_transformed[f'AWAY_AVG{self.hist_avg_window}_OREB_PCT']

        self.df_transformed.fillna(0, inplace=True)

    def run_transform(self):
        # Step 1 - Join Stage 1 Results to Stage 1 Transformed Data
        self.merge_stage1_results()

        # Step 2 - Clean data for stage 2 join + features
        self.clean_data()

        # Step 3 - Create historical features
        self.create_historical_features()

        # Step 4 - Join historical features
        self.join_historical_features()

        # Step 5 - Split data
        self.df_train = self.df_transformed[self.df_transformed['GAME_DATE'] < self.current_date].copy()
        self.df_test = self.df_transformed[self.df_transformed['GAME_DATE'] == self.current_date].copy()

In [3]:
class ModelStage2(RegressionDiagnostics):
    """
    This class trains the stage 2 model to output predicted net rating for 
    the upcoming game given features created from past games. 
    """
    def __init__(
            self, 
            X_train: pd.DataFrame, 
            y_train: pd.Series, 
            X_test: pd.DataFrame, 
            model_name: str, 
            target_name: str, 
            id_cols: List[str],
            current_date: str
        ) -> None:
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.reg_stage2: Optional[RegressionResultsWrapper] = None
        self.model_name = model_name
        self.target_name = target_name
        self.current_date = current_date
        self.id_cols = id_cols
        if self.id_cols:
            self.id_data_train = X_train[self.id_cols].copy() # store ID columns separately
            self.id_data_test = X_test[self.id_cols].copy()
            self.X_train = X_train.drop(self.id_cols, axis=1) # features for modeling
            self.X_test = X_test.drop(self.id_cols, axis=1)
        else:
            self.id_data_train = pd.DataFrame(index=X_train.index)
            self.X_train = X_train.copy()
            self.id_data_test = pd.DataFrame(index=X_test.index)
            self.X_test = X_test.copy()

    def fit_stage2_model(self) -> RegressionResultsWrapper:
        """Fit Stage 2 OLS Model"""
        logger.info(f"Fitting stage 2 model...")
        X1 = sm.add_constant(self.X_train)
        self.reg_stage2 = sm.OLS(endog=self.y_train, exog=X1).fit()
        
        return self.reg_stage2
    
    def predict_stage2_model(self):
        """Predict Net Rating for Upcoming Games"""
        if self.reg_stage2 is None:
            raise ValueError("Stage 2 model must be fitted first. Call fit_stage2_model()") 
                         
        logger.info(f"Predicting stage 2 model upcoming games...")
        X1 = sm.add_constant(self.X_test)
        
        return self.reg_stage2.predict(X1)
    
    def _print_model_summary(self):
        """Print Model Summary and RMSE"""
        if self.reg_stage2 is None:
            raise ValueError("Stage 2 model must be fitted first")
        
        print(self.reg_stage2.summary())

    def run_stage2_model(self, output_dir: str, save_figs: bool =False, print_output: bool =False, create_plots: bool =False):
        """Run Stage 2 Model"""
        # Fit model
        model = self.fit_stage2_model()

        # Print statsmodels output
        if print_output:
            self._print_model_summary()

        # Training Predictions
        y_pred_train = model.fittedvalues
        y_pred_train_df = pd.DataFrame({'y_pred_train': y_pred_train})
        df_train = pd.concat([self.X_train, y_pred_train_df], axis=1)
        logger.info(f"Train RMSE: {rmse(y_true=self.y_train, y_pred=y_pred_train): .3f}")
        print("="*50)
        print("Train Data")
        print("="*50)
        print(df_train.head())
        print("="*50)
        

        # Create OLS diagnostic plots
        if create_plots:
            self.create_diagnostic_plots(
                model_name=self.model_name,
                target_name=self.target_name,
                y_true=self.y_train, 
                y_pred=y_pred_train, 
                save_figs=save_figs
            )

        # Predict upcoming games
        y_pred_test_df = pd.DataFrame({'y_pred_test': self.predict_stage2_model()})
        df_test = pd.concat([self.X_test, y_pred_test_df], axis=1)
        print("="*50)
        print("Test Data")
        print("="*50)
        print(df_test.head())
        

        # Save
        current_date_fmt = datetime.strftime(datetime.strptime(self.current_date,"%Y-%m-%d"),"%Y%m%d")
        os.makedirs(output_dir, exist_ok=True)
        train_output_path = os.path.join(output_dir, f'df_train_stage2_effort_{current_date_fmt}.csv')
        test_output_path = os.path.join(output_dir, f'df_test_stage2_effort_{current_date_fmt}')

        df_test.to_csv(test_output_path, index=False)
        df_train.to_csv(train_output_path, index=False)

In [4]:
current_date='2024-10-22'
window=10
DATA_DIR = '../data/'
transformed_path = os.path.join(DATA_DIR, 'transformed_data', 'df_transformed.csv')
df_trans = pd.read_csv(transformed_path)
stage1_path = os.path.join(DATA_DIR, 'stage1_effort', 'df_net_stage1_effort.csv')
df_stage1 = pd.read_csv(stage1_path)
output_dir = os.path.join(DATA_DIR, 'stage2_netrating')

# Transform Data for Stage 2 Model
trans_data = TransformStage2(
    df_transformed=df_trans,
    df_stage1_results=df_stage1,
    current_date=current_date,
    hist_avg_window=window
)
trans_data.run_transform()
ids = ['SEASON_ID','GAME_ID','GAME_DATE','HOME_TEAM_ID','AWAY_TEAM_ID','HOME_TEAM','AWAY_TEAM'] 
# Prep columns and target
target = 'EST_HOME_NRtg'
X_cols = ids + [
    f'AVG{window}_EFG_PCT_DIFF',
    f'AVG{window}_FTA_RATE_DIFF',
    f'AVG{window}_TM_TOV_PCT_DIFF',
    f'AVG{window}_OREB_PCT_DIFF',
    f'AVG{window}_NET_COMPOSITE_EFFORT_DIFF'
]
#print(trans_data.df_train.head())
#print(trans_data.df_train.columns[:20])

y_train = trans_data.df_train[target]
X_train = trans_data.df_train[X_cols]

y_test = trans_data.df_test[target]
X_test = trans_data.df_test[X_cols]


model_nrtg = ModelStage2(
    X_train=X_train, y_train=y_train, 
    X_test=X_test, 
    model_name="Stage 2 Net Rating", 
    target_name=target, 
    id_cols=ids, 
    current_date=current_date
)
model_nrtg.run_stage2_model(
    output_dir=output_dir,
    save_figs=False,
    print_output=True,
    create_plots=False
)

2025-09-21 16:06:37,088 [INFO] Fitting stage 2 model...
2025-09-21 16:06:37,095 [INFO] Train RMSE:  14.425
2025-09-21 16:06:37,098 [INFO] Predicting stage 2 model upcoming games...


                            OLS Regression Results                            
Dep. Variable:          EST_HOME_NRtg   R-squared:                       0.075
Model:                            OLS   Adj. R-squared:                  0.073
Method:                 Least Squares   F-statistic:                     39.71
Date:                Sun, 21 Sep 2025   Prob (F-statistic):           2.33e-39
Time:                        16:06:37   Log-Likelihood:                -10056.
No. Observations:                2460   AIC:                         2.012e+04
Df Residuals:                    2454   BIC:                         2.016e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

In [5]:
model_nrtg.X_train.shape, model_nrtg.X_test.shape

((2460, 5), (2, 5))

In [6]:
display(model_nrtg.X_train.head())
display(model_nrtg.X_test.head())

,AVG10_EFG_PCT_DIFF,AVG10_FTA_RATE_DIFF,AVG10_TM_TOV_PCT_DIFF,AVG10_OREB_PCT_DIFF,AVG10_NET_COMPOSITE_EFFORT_DIFF
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0


,AVG10_EFG_PCT_DIFF,AVG10_FTA_RATE_DIFF,AVG10_TM_TOV_PCT_DIFF,AVG10_OREB_PCT_DIFF,AVG10_NET_COMPOSITE_EFFORT_DIFF
2460,0.0222,-0.0866,-0.0121,-0.0438,-1.202606
2461,0.0177,-0.0182,0.0120,-0.0067,-1.233607


In [7]:
model_nrtg.id_data_train

,SEASON_ID,GAME_ID,GAME_DATE,HOME_TEAM_ID,AWAY_TEAM_ID,HOME_TEAM,AWAY_TEAM
0,22022,22200001,2022-10-18,1610612738,1610612755,BOS,PHI
1,22022,22200002,2022-10-18,1610612744,1610612747,GSW,LAL
2,22022,22200003,2022-10-19,1610612765,1610612753,DET,ORL
3,22022,22200004,2022-10-19,1610612754,1610612764,IND,WAS
4,22022,22200005,2022-10-19,1610612737,1610612745,ATL,HOU
...,...,...,...,...,...,...,...
2455,22023,22301226,2023-12-08,1610612757,1610612742,POR,DAL
2456,22023,22301227,2023-12-08,1610612738,1610612752,BOS,NYK
2457,22023,22301228,2023-12-08,1610612756,1610612758,PHX,SAC
2458,22023,22301229,2023-12-07,1610612749,1610612754,MIL,IND


In [8]:
model_nrtg.id_data_train['GAME_DATE'].min(), model_nrtg.id_data_train['GAME_DATE'].max()

(Timestamp('2022-10-18 00:00:00'), Timestamp('2024-04-14 00:00:00'))

In [9]:
model_nrtg.id_data_test

,SEASON_ID,GAME_ID,GAME_DATE,HOME_TEAM_ID,AWAY_TEAM_ID,HOME_TEAM,AWAY_TEAM
2460,22024,22400061,2024-10-22,1610612738,1610612752,BOS,NYK
2461,22024,22400062,2024-10-22,1610612747,1610612750,LAL,MIN


In [10]:
X_test.head()

,SEASON_ID,GAME_ID,GAME_DATE,HOME_TEAM_ID,AWAY_TEAM_ID,HOME_TEAM,AWAY_TEAM,AVG10_EFG_PCT_DIFF,AVG10_FTA_RATE_DIFF,AVG10_TM_TOV_PCT_DIFF,AVG10_OREB_PCT_DIFF,AVG10_NET_COMPOSITE_EFFORT_DIFF
2460,22024,22400061,2024-10-22,1610612738,1610612752,BOS,NYK,0.0222,-0.0866,-0.0121,-0.0438,-1.202606
2461,22024,22400062,2024-10-22,1610612747,1610612750,LAL,MIN,0.0177,-0.0182,0.0120,-0.0067,-1.233607


In [11]:
x1 = sm.add_constant(model_nrtg.X_test)

In [12]:
x1

,const,AVG10_EFG_PCT_DIFF,AVG10_FTA_RATE_DIFF,AVG10_TM_TOV_PCT_DIFF,AVG10_OREB_PCT_DIFF,AVG10_NET_COMPOSITE_EFFORT_DIFF
2460,1.0,0.0222,-0.0866,-0.0121,-0.0438,-1.202606
2461,1.0,0.0177,-0.0182,0.0120,-0.0067,-1.233607


In [13]:
y_pred_test_df = pd.DataFrame({'y_pred_test': model_nrtg.reg_stage2.predict(x1)})

In [14]:
y_pred_test_df

,y_pred_test
2460,3.278085
2461,2.154545


In [15]:
d_test = pd.concat([model_nrtg.X_test, y_pred_test_df], axis=1)

In [16]:
d_test

,AVG10_EFG_PCT_DIFF,AVG10_FTA_RATE_DIFF,AVG10_TM_TOV_PCT_DIFF,AVG10_OREB_PCT_DIFF,AVG10_NET_COMPOSITE_EFFORT_DIFF,y_pred_test
2460,0.0222,-0.0866,-0.0121,-0.0438,-1.202606,3.278085
2461,0.0177,-0.0182,0.0120,-0.0067,-1.233607,2.154545
